# NewsQA RAG - Phase 1 Retrieval Tournament (Kaggle)
Resumable GPU tournament. Original questions select winners; resolved questions are supplementary paired analysis.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile, time
REPO_URL = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = '62d7e200ca685963e64931863eb2e9a4b047eb16'
HF_REPO_ID = 'MatchaMacchiato/newsqa_200_11064_v2.0.0'
HF_REVISION = 'b81c8db6847a23272665946c0c43c72e9a212fd9'  # the v2.0.0 commit; swap for 'v2.0.0' once that tag exists
SMOKE_MODE = True
SMOKE_QUESTIONS = 5
FAST_MODE = SMOKE_MODE
GPU_COUNT = 2
RUN_LATENCY_CALIBRATION = True
LATENCY_REPEATS, LATENCY_QUESTIONS = 3, 100
KEEP_STAGE_CHECKPOINTS = True
AUTO_RESTORE_FROM_INPUT = True
RESTORE_CHECKPOINT = ''
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT_ROOT = KAGGLE_WORKING / 'Text-Mining---NewsQA-RAG'
WORK_ROOT = KAGGLE_WORKING / ('newsqa_phase1_smoke' if SMOKE_MODE else 'newsqa_phase1')
RESULTS = WORK_ROOT / 'results'
CHECKPOINT_PATH = KAGGLE_WORKING / f'{WORK_ROOT.name}_checkpoint.tar'
assert REPO_COMMIT != 'SET_TO_COMMIT_CONTAINING_KAGGLE_RUNNER'
assert GPU_COUNT in {1, 2}


## 1. Secret, Kaggle input, repository and GPUs
Add `HF_TOKEN` under Add-ons > Secrets. Enable Internet and select a GPU accelerator. To resume a previous saved version, attach its output as a Kaggle input; the newest compatible checkpoint is restored automatically.

In [ ]:
# The dataset is public, so a token is optional. Set one as the Kaggle
# secret HF_TOKEN only to lift anonymous download rate limits.
token=''
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    pass
if token:
    os.environ['HF_TOKEN']=token
else:
    print('No HF_TOKEN secret; downloading the public dataset anonymously.')
os.environ['HF_HOME']=str(KAGGLE_WORKING/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})
if AUTO_RESTORE_FROM_INPUT and not RESTORE_CHECKPOINT and not WORK_ROOT.exists() and KAGGLE_INPUT.exists():
    candidates=list(KAGGLE_INPUT.rglob(f'{WORK_ROOT.name}_checkpoint.tar'))
    candidates+=list(KAGGLE_INPUT.rglob(f'{WORK_ROOT.name}_round*_checkpoint.tar'))
    candidates.sort(key=lambda p:p.stat().st_mtime,reverse=True)
    for candidate in candidates:
        if tarfile.is_tarfile(candidate):
            RESTORE_CHECKPOINT=str(candidate); break
    print('Auto-restore checkpoint:',RESTORE_CHECKPOINT or 'none found')
if RESTORE_CHECKPOINT and not WORK_ROOT.exists():
    WORK_ROOT.mkdir(parents=True,exist_ok=True)
    print('Restoring indexed state from:',RESTORE_CHECKPOINT,flush=True)
    shutil.unpack_archive(RESTORE_CHECKPOINT,WORK_ROOT)
    print('Restored index manifests:',len(list((WORK_ROOT/'indexes').rglob('index_manifest.json'))),flush=True)
    print('Restored completed reports:',len(list((WORK_ROOT/'experiments').rglob('report.json'))),flush=True)
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','--depth','30',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Notebook options'
assert torch.cuda.device_count() >= GPU_COUNT, f'GPU_COUNT={GPU_COUNT}, but only {torch.cuda.device_count()} GPU(s) are available'
for i in range(GPU_COUNT):
    print(i,torch.cuda.get_device_name(i),round(torch.cuda.get_device_properties(i).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())


In [ ]:
import pandas as pd
from IPython.display import display, Image
def ensure_fresh_checkpoint(started_at):
    if not WORK_ROOT.exists(): return
    if CHECKPOINT_PATH.exists() and CHECKPOINT_PATH.stat().st_mtime >= started_at: return
    print('Creating recovery checkpoint from current data and indexes...',flush=True)
    shutil.make_archive(str(CHECKPOINT_PATH.with_suffix('')),'tar',WORK_ROOT,base_dir='.')
def preserve_outputs(stage=None,started_at=0):
    ensure_fresh_checkpoint(started_at)
    if not CHECKPOINT_PATH.exists(): return
    print('Checkpoint:',CHECKPOINT_PATH,round(CHECKPOINT_PATH.stat().st_size/2**20,1),'MiB',flush=True)
    if stage and KEEP_STAGE_CHECKPOINTS:
        stage_path=KAGGLE_WORKING/f'{WORK_ROOT.name}_{stage}_checkpoint.tar'
        shutil.copy2(CHECKPOINT_PATH,stage_path)
        print('Stage snapshot:',stage_path,round(stage_path.stat().st_size/2**20,1),'MiB',flush=True)
def run_driver(stage):
    cmd=[sys.executable,'-u','scripts/run_phase1_kaggle.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--work-root',str(WORK_ROOT),'--stop-after',stage,'--gpu-count',str(GPU_COUNT),'--checkpoint-path',str(CHECKPOINT_PATH)]
    if FAST_MODE: cmd.append('--fast')
    if SMOKE_MODE: cmd += ['--smoke-questions',str(SMOKE_QUESTIONS)]
    if RESTORE_CHECKPOINT and stage=='round1' and not WORK_ROOT.exists(): cmd += ['--restore-checkpoint',RESTORE_CHECKPOINT]
    log_path=WORK_ROOT/'logs'/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    log_path.parent.mkdir(parents=True,exist_ok=True)
    print('$',' '.join(cmd),flush=True)
    print('Detailed log:',log_path,flush=True)
    started_at=time.time(); succeeded=False
    try:
        with log_path.open('w',encoding='utf-8') as log:
            process=subprocess.Popen(cmd,cwd=PROJECT_ROOT,env={**os.environ,'PYTHONUNBUFFERED':'1'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush()
            returncode=process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode,cmd)
        succeeded=True
    except subprocess.CalledProcessError as error:
        print(f'FAILED stage={stage} exit_code={error.returncode}',flush=True)
        print('Inspect the detailed log at:',log_path,flush=True)
        raise
    finally:
        latest_log=KAGGLE_WORKING/f'{WORK_ROOT.name}_{stage}_latest.log'
        if log_path.exists():
            shutil.copy2(log_path,latest_log); print('Latest log:',latest_log,flush=True)
        snapshot_stage=stage if succeeded else f'{stage}_interrupted'
        preserve_outputs(snapshot_stage,started_at)
def show_csv(name,sort='retrieval.mrr@5.mean'):
    frame=pd.read_csv(RESULTS/name)
    if sort in frame: frame=frame.sort_values(sort,ascending=False)
    display(frame); return frame
def paired(frame):
    keys=[k for k in ['index','retriever','reranker','reranker_model','partition'] if k in frame]
    values=[k for k in ['retrieval.mrr@5.mean','retrieval.hit_rate@5.mean','retrieval.ndcg@5.mean'] if k in frame]
    return frame.pivot_table(index=keys,columns='variant',values=values).reset_index()


## 2. Dataset preparation and Round 1
Lightweight GPU and CPU indexes may build concurrently. Host-memory-heavy models are isolated and built sequentially.

In [ ]:
started=time.time(); run_driver('round1')
round1=show_csv('round1.csv'); display(paired(round1))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'round1_winners.json').read_text()))


## 3. Round 2: retrieval methods and rerankers
Large reranker experiments are isolated to reduce host-memory pressure.

In [ ]:
started=time.time(); run_driver('round2')
round2=show_csv('round2.csv'); display(paired(round2))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 4. Round 3: chunking ablation
Chunk profiles use the locked Round 2 retrieval configuration.

In [ ]:
started=time.time(); run_driver('round3')
round3=show_csv('round3.csv'); display(paired(round3))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 5. Locked final-test evaluation

In [ ]:
started=time.time(); run_driver('final')
final_test=show_csv('final_test.csv'); display(paired(final_test))
print('Minutes:',round((time.time()-started)/60,1))


## 6. Serial cache-free latency calibration

In [ ]:
if RUN_LATENCY_CALIBRATION and not FAST_MODE:
    specs=sorted((WORK_ROOT/'specs').glob('round*.yaml'))
    cmd=[sys.executable,'-u','scripts/calibrate_phase1_latency.py',*map(str,specs),'--output',str(RESULTS/'latency_calibration.csv'),'--repeats',str(LATENCY_REPEATS),'--n-eval',str(LATENCY_QUESTIONS)]
    subprocess.run(cmd,cwd=PROJECT_ROOT,check=True,env={**os.environ,'CUDA_VISIBLE_DEVICES':'0','PYTHONUNBUFFERED':'1'})
    show_csv('latency_calibration.csv',sort='latency.total.p50_ms'); preserve_outputs()
else: print('Latency calibration skipped')


## 7. Figures, archives and Kaggle outputs
After Save Version > Save & Run All completes, download these files from the notebook Output tab. Attach a checkpoint TAR as an input to resume in a later Kaggle session.

In [ ]:
subprocess.run([sys.executable,'-u','scripts/export_phase1_results.py','--experiments-root',str(WORK_ROOT/'experiments'),'--output-dir',str(RESULTS)],cwd=PROJECT_ROOT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})
preserve_outputs()
for figure in sorted((RESULTS/'figures').glob('*.png')):
    print(figure.name); display(Image(filename=str(figure)))
for path in sorted(RESULTS.rglob('*')):
    if path.is_file(): print(path.relative_to(RESULTS),round(path.stat().st_size/2**20,2),'MiB')
print('Results archive:',WORK_ROOT/'phase1_results_bundle.zip')
print('Checkpoint:',CHECKPOINT_PATH)
print('Stage checkpoints and latest logs:',KAGGLE_WORKING)
